# Installation PyPSA-Eur

This notebook walks through the full PyPSA-Eur workflow: setup, running the model, and analysing results from a solved sector-coupled network.

## Part 0 - Import PyPSA-Eur

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
import platform

if "__file__" in globals():  # paths relative to the script folder, wherever it is launched from
    os.chdir(Path(__file__).resolve().parent)

# PyPSA-Eur in a SHORT path: Windows limits paths to 260 characters and Snakemake
# creates files with very long names in .snakemake/ -> failure in a deeply nested folder.
pypsa_eur_dir = pypsa_eur_dir = Path("pypsa-eur")  # relative path to the script folder
pypsa_eur_file = pypsa_eur_dir / "resources/networks/base_s_1_elec.nc"
# Pre-generated copy shipped with the tutorial, in case the workflow below fails
backup_file = Path("pypsa-eur/resources/networks/base_s_1_elec.nc").resolve()

PYPSA_EUR_CONFIG = """\
benchmark:
  enabled: false

countries:
  - FR

scenario:
  clusters: [1]
  opts: [""]
  sector_opts: [""]
  planning_horizons: [2050]

snapshots:
  start: "2013-01-01"
  end: "2013-01-03"
  inclusive: left

solving:
  solver:
    name: highs
    options: "highs-default"
  mem_mb: 10000

atlite:
  nprocesses: 2

sector:
  hydrogen_underground_storage_locations:
    - onshore
  regional_co2_sequestration_potential:
    enable: false
  district_heating:
    supply_temperature_approximation:
      rolling_window_ambient_temperature: 48

plotting:
  costs_threshold: 0.001
  energy_threshold: 0.1
"""

if not pypsa_eur_file.is_file():
    try:
        # 0.1 Install Pixi (PyPSA-Eur package manager)
        pixi_exe = "pixi.exe" if platform.system() == "Windows" else "pixi"
        pixi = shutil.which("pixi") or str(Path.home() / ".pixi" / "bin" / pixi_exe)
        if not Path(pixi).is_file():
            if platform.system() == "Windows":
                subprocess.run(["powershell", "-ExecutionPolicy", "ByPass", "-c",
                            "irm -useb https://pixi.sh/install.ps1 | iex"], check=True)
            elif platform.system() == "Linux":
                subprocess.run(["bash", "-c",
                            "curl -sSL https://pixi.sh/install.sh | bash"], check=True)
            else:
                raise OSError(f"Pixi installation not implemented for {platform.system()}")
        else:
            # An already-installed Pixi may be too old for the PyPSA-Eur manifest
            # (e.g. pixi 0.66 rejects `exclude-newer = "7d"` in pixi.toml)
            subprocess.run([pixi, "self-update"], check=False)
        print(f"Pixi installed: {pixi}")

        # 0.2 Clone PyPSA-Eur v2026.08.0: last release that still produces base_s_{clusters}_elec.nc
        #     (from v2026.09.0 on, networks are named networks/clustered.nc / composed_{horizon}.nc)
        if not pypsa_eur_dir.is_dir():
            subprocess.run(["git", "clone", "--branch", "v2026.08.0", "--depth", "1",
                            "https://github.com/PyPSA/pypsa-eur.git", str(pypsa_eur_dir)], check=True)
        print(f"PyPSA-Eur cloned: {pypsa_eur_dir}")

        # 0.3 Install the environment (reads pixi.toml -> .pixi/envs/default/)
        subprocess.run([pixi, "install"], cwd=pypsa_eur_dir, check=True)
        print(f"Environment installed: {pypsa_eur_dir}")

        # 0.4 Configure the study (FR, 1 cluster, 48 h); automatically merged with config.default.yaml
        (pypsa_eur_dir / "config/config.yaml").write_text(PYPSA_EUR_CONFIG, encoding="utf-8")
        print(f"PyPSA-Eur configured: {pypsa_eur_dir / 'config/config.yaml'}")

        # 0.5 Run the workflow only up to base_s_1_elec.nc
        #     (no sector coupling, no solving; --rerun-incomplete resumes after an interruption)
        subprocess.run([pixi, "run", "snakemake", "-c", "all", "-j", "4", "--rerun-incomplete",
                        pypsa_eur_file.relative_to(pypsa_eur_dir).as_posix()],
                       cwd=pypsa_eur_dir, check=True)
        print(f"PyPSA-Eur generated: {pypsa_eur_file}")
        
    except (subprocess.CalledProcessError, OSError) as err:
        print(f"PyPSA-Eur generation failed: {err}")
else:
    print(f"PyPSA-Eur already generated: {pypsa_eur_file}")

✔ pixi is already up-to-date (version 0.84.2)
 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the top-level `[exclude-newer]` table of per-package cutoffs is deprecated
  │ in favor of the `exemptions` of `[workspace.exclude-newer]`
    ╭─[/home/gmaistre/pypsa-eur/pypsa-eur/pixi.toml:13:1]
 12 │     
 13 │ ╭─▶ [exclude-newer]
 14 │ │   linopy = "0d"
 15 │ │   pypsa = "0d"
 16 │ │   atlite = "0d"
 17 │ ├─▶ snakemake-executor-plugin-slurm = "0d"  # drop once the pandas 3 build is older than 7d
    · ╰──── replace this with `[workspace.exclude-newer.exemptions]`
 18 │     
    ╰────
  help: move the entries to the `exemptions` of `[workspace.exclude-newer]`
        instead, e.g. replace the current `exclude-newer` of `[workspace]`
        with
        
        [workspace.exclude-newer]
        cutoff = "7days"
        exemptions = { linopy = "*", pypsa = "*", atlite = "*", snakemake-
        executor-plugin-slurm = "*" }
        
        An exemption allows the matching releas

Pixi installed: /home/gmaistre/.pixi/bin/pixi
PyPSA-Eur cloned: /home/gmaistre/pypsa-eur/pypsa-eur


✔ The default environment has been installed.
 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the top-level `[exclude-newer]` table of per-package cutoffs is deprecated
  │ in favor of the `exemptions` of `[workspace.exclude-newer]`
    ╭─[/home/gmaistre/pypsa-eur/pypsa-eur/pixi.toml:13:1]
 12 │     
 13 │ ╭─▶ [exclude-newer]
 14 │ │   linopy = "0d"
 15 │ │   pypsa = "0d"
 16 │ │   atlite = "0d"
 17 │ ├─▶ snakemake-executor-plugin-slurm = "0d"  # drop once the pandas 3 build is older than 7d
    · ╰──── replace this with `[workspace.exclude-newer.exemptions]`
 18 │     
    ╰────
  help: move the entries to the `exemptions` of `[workspace.exclude-newer]`
        instead, e.g. replace the current `exclude-newer` of `[workspace]`
        with
        
        [workspace.exclude-newer]
        cutoff = "7days"
        exemptions = { linopy = "*", pypsa = "*", atlite = "*", snakemake-
        executor-plugin-slurm = "*" }
        
        An exemption allows the matching releas

Environment installed: /home/gmaistre/pypsa-eur/pypsa-eur
PyPSA-Eur configured: /home/gmaistre/pypsa-eur/pypsa-eur/config/config.yaml


Using workflow specific profile profiles/default for setting default command line arguments.
Assuming unrestricted shared filesystem usage.

SNAKEMAKE
  Date: 2026-11-13 15:19:52
  Workflow ID: 16619864-d1e3-415f-b15d-3d1ac8eedb93
  Platform: Linux-6.8.0-147-generic-x86_64-with-glibc2.39
  Host: RTEI078
  User: gmaistre
  Snakemake version: 9.25.1
  Python version: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]
  Command: /home/gmaistre/pypsa-eur/pypsa-eur/.pixi/envs/default/bin/snakemake -c all -j 4 --rerun-incomplete resources/networks/base_s_1_elec.nc
  Snakefile: /home/gmaistre/pypsa-eur/pypsa-eur/Snakefile
  Base directory: /home/gmaistre/pypsa-eur/pypsa-eur
  Run directory: /home/gmaistre/pypsa-eur/pypsa-eur
  Working directory: /home/gmaistre/pypsa-eur/pypsa-eur
  Config file(s): ['config/config.default.yaml', 'config/plotting.default.yaml', 'config/config.yaml']
  Config MD5: 57a2cded252c9fd9968b4f1f4ad172bf

Building DAG of jobs...
Using shell: 

KeyboardInterrupt: 

[Errno 28] No space left on device: '/home/gmaistre/.cache/snakemake-pypsa-eur/data.pypsa.org/workflows/cutout/v1.0/europe-2013-sarah3-era5.nc' -> '.snakemake/storage/cached-http/data.pypsa.orgworkflows/cutout/v1.0/europe-2013-sarah3-era5.nc', attempt 4/5 failed - retrying in 24 seconds...
Terminating processes on user request, this might take some time.
WorkflowError:
At least one job did not complete successfully.
[Fri Nov 13 15:48:04 2026]
Error in rule retrieve_electricity_demand_energy_atlas:
    message: None
    jobid: 25
    output: data/jrc_energy_atlas/archive/2026-02-16/electricity_tot_demand_2019.tif
Complete log(s): /home/gmaistre/pypsa-eur/pypsa-eur/.snakemake/log/2026-11-13T151946.817946.snakemake.log
WorkflowError:
At least one job did not complete successfully.


## Part 1 - Loading the network

### 1.1 Loading the network

Results are stored as NetCDF files under `pypsa-eur/results/networks/`.
The filename encodes the configuration: `base_s_***.nc`

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pypsa
import pandas as pd
import matplotlib.pyplot as plt

n = pypsa.Network("pypsa-eur/resources/networks/base_s_1_elec.nc")
print(n)

# Conversion to GEMS format

## Part 2 - Imports & Logging

Firstly we import the Python package ; [pypsa-to-gems-converter](https://pypi.org/project/pypsa-to-gems-converter/)

In [ ]:
import sys
import src as _pypsa_to_gems_converter_pkg
sys.modules.setdefault("pypsa_to_gems_converter", _pypsa_to_gems_converter_pkg)

from pypsa_to_gems_converter import PyPSAStudyConverter
import yaml
import logging
from pathlib import Path
import pypsa
from pypsa import Network
import importlib.metadata


# Configure logging so progress is visible in the notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s – %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("pypsa_to_gems")

study_dir = Path("tmp/my_study")  # Absolute path to the GEMS study directory

PyPSA_file_path = "pypsa-eur/resources/networks/base_s_1_elec.nc"  # Absolute path to the PyPSA file
network = Network(PyPSA_file_path)
print(network)  # Print the network to verify it loaded correctly

## Part 3 - Conversion PYPSA network to GEMS

In [ ]:
# Convert PyPSA network to GEMS
PyPSAStudyConverter(
    pypsa_network=network,
    study_dir=study_dir,
    series_file_format=".tsv",  # Supported formats: .tsv, .csv
    solver_name="coin",  # Investment studies only support 'coin' and 'xpress'
).to_gems_study()

## Part 4 - Inspect the GEMS Study

In [ ]:
import yaml
from collections import defaultdict

gems_study_file_path = Path(study_dir) / "systems" / "input"

system_yml_files = list(gems_study_file_path.rglob("system.yml"))
if not system_yml_files:
    print("system.yml not found.")
else:
    data = yaml.safe_load(system_yml_files[0].read_text())["system"]

    components = data.get("components", [])
    connections = data.get("connections", [])

    # Group components by model type
    by_model = defaultdict(list)
    for c in components:
        by_model[c["model"]].append(c["id"])

    print(f"System : {data['id']}")
    print(f"\nComponents ({len(components)} total)")
    print("-" * 20)
    for model, ids in sorted(by_model.items()):
        short = model.split(".")[-1]
        print(f"  [{short}]  ({len(ids)})")
        for cid in ids:
            print(f"    - {cid}")

    print(f"\nConnections ({len(connections)} total)")
    print("-" * 20)
    for conn in connections:
        print(f"  {conn.get('component1')}.{conn.get('port1')}  →  {conn.get('component2')}.{conn.get('port2')}")

## Part 5 - Run the GEMS optimization

In this section, we use [GemsPy](https://gemspy.readthedocs.io/en/latest/) to run the GEMS converted simulation.

In [ ]:
from gems_craft.study.folder import load_study
from gems_runner.session.session import SimulationSession
from gems_craft.optim_config.parsing import OptimConfig, TimeScopeConfig, ScenarioScopeConfig

n_timesteps = len(network.snapshots)

print("STUDY LOADING")
study = load_study(study_dir / "systems")
print("\tStudy loaded")

optim_config = OptimConfig(
    time_scope=TimeScopeConfig(first_time_step=0, last_time_step=n_timesteps - 1),
    scenario_scope=ScenarioScopeConfig(include=[0]),
)

print("\nSOLVING OPTIMIZATION PROBLEM")
result = SimulationSession(study=study, optim_config=optim_config).run()
print("\tOptimization problem solved")

## Part 6 - Analyse the simulation results

The results are returned as a `SimulationTable` object by GemsPy.  
Below we extract the long-format DataFrame and visualise:
- the **objective value** (total system cost)
- the **installed capacity** (`p_nom`) per generator technology
- the **dispatch** (`p`) time series per technology

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = result.data

# ── Objective value ────────────────────────────────────────────────────────────
obj_row = df[df["output"] == "objective-value"]
if not obj_row.empty:
    obj_value = float(obj_row["value"].iloc[0])
    print(f"Objective value (total system cost): {obj_value:,.0f} €")

# ── Installed capacity (p_nom) per generator technology ───────────────────────
p_nom_df = df[
    (df["output"] == "p_nom") &
    (df["component"].str.startswith("generator_", na=False))
].copy()

p_nom_df["technology"] = p_nom_df["component"].str.split("_").str[-1]
capacity = p_nom_df.groupby("technology")["value"].sum().sort_values(ascending=False)
capacity = capacity[capacity > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(capacity.index, capacity.values / 1e3, color="steelblue")
axes[0].set_xlabel("Installed capacity (GW)")
axes[0].set_title("Generator installed capacity (p_nom)")
axes[0].invert_yaxis()

# ── Dispatch time series (p) per technology ───────────────────────────────────
p_df = df[
    (df["output"] == "p") &
    (df["component"].str.startswith("generator_", na=False)) &
    (df["absolute_time_index"].notna())
].copy()

p_df["technology"] = p_df["component"].str.split("_").str[-1]
p_df["t"] = pd.to_numeric(p_df["absolute_time_index"])

dispatch = (
    p_df.groupby(["t", "technology"])["value"]
    .sum()
    .unstack("technology")
    .fillna(0)
    .clip(lower=0)
)
dispatch = dispatch / 1e3

dispatch.plot.area(ax=axes[1], linewidth=0)
axes[1].set_xlabel("Time step")
axes[1].set_ylabel("Generation (GW)")
axes[1].set_title("Dispatch by technology")
axes[1].legend(loc="upper right", fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

## Part 7 - Validate the transfer

To verify that the conversion is faithful, we solve the original PyPSA network and compare it with the GEMS result: total cost first, then the optimal capacity per carrier to localise any gap.

In [ ]:
network.optimize(solver_name="highs")
# PyPSA keeps the fixed cost of existing non-extendable capacity apart (objective_constant); GEMS includes it.
pypsa_obj = network.objective + network.objective_constant
print(f"GEMS total cost  : {obj_value:,.0f} €")
print(f"PyPSA total cost : {pypsa_obj:,.0f} €")
print(f"Relative gap     : {abs(obj_value - pypsa_obj) / pypsa_obj:.1e}")

# Per-carrier comparison of the optimal capacities (localises any gap in the objective)
pypsa_cap = network.generators.groupby("carrier")["p_nom_opt"].sum()
gems_cap = (df[(df["output"] == "p_nom") & df["component"].str.startswith("generator_", na=False)]
            .assign(carrier=lambda d: d["component"].str.split("_").str[-1])
            .groupby("carrier")["value"].sum())
cap_cmp = pd.concat({"PyPSA (MW)": pypsa_cap, "GEMS (MW)": gems_cap}, axis=1).fillna(0)
cap_cmp["abs diff (MW)"] = (cap_cmp["PyPSA (MW)"] - cap_cmp["GEMS (MW)"]).abs()
print(cap_cmp.round(1))
print("Max capacity difference (MW):", round(cap_cmp["abs diff (MW)"].max(), 3))